## Original Algorithm - 5-Node Mesh

In [5]:
import numpy as np
import netsquid as ns
from netsquid.nodes import Node
from netsquid.components import QuantumChannel
from netsquid.components.models.qerrormodels import DepolarNoiseModel
import netsquid.qubits.operators as ops

# =====================================================================
# PHYSICAL LINK PROBER (Fixed NetSquid Hardware Abstraction)
# =====================================================================
def probe_physical_link(noise_rate, num_trials=60):
    """Simulates physical link layer execution to return an empirical S-value."""
    ns.sim_reset()
    n1, n2 = Node("Tx"), Node("Rx")
    noise_model = DepolarNoiseModel(depolar_rate=noise_rate, time_independent=True)
    channel = QuantumChannel("Link", length=1.0, models={"quantum_noise_model": noise_model})
    
    n1.add_ports(["out"])
    n2.add_ports(["in"])
    n1.ports["out"].connect(channel.ports["send"])
    channel.ports["recv"].connect(n2.ports["in"])
    
    products = []
    alice_angles = [0.0, np.pi / 4]
    bob_angles   = [np.pi / 8, -np.pi / 8]
    
    for _ in range(num_trials):
        qA, qB = ns.qubits.create_qubits(2)
        ns.qubits.operate(qA, ops.H)
        ns.qubits.operate([qA, qB], ops.CNOT)
        n1.ports["out"].tx_output(qB)
        ns.sim_run()
        
        qB_rx = n2.ports["in"].rx_input().items[0]
        idx_A, idx_B = np.random.choice([0, 1]), np.random.choice([0, 1])
        
        # Realignment operators inline
        c_A, s_A = np.cos(-alice_angles[idx_A]), np.sin(-alice_angles[idx_A])
        c_B, s_B = np.cos(-bob_angles[idx_B]), np.sin(-bob_angles[idx_B])
        
        ns.qubits.operate(qA, ops.Operator("RotA", np.array([[c_A, -s_A], [s_A, c_A]])))
        ns.qubits.operate(qB_rx, ops.Operator("RotB", np.array([[c_B, -s_B], [s_B, c_B]])))
        
        val_A, _ = ns.qubits.measure(qA, ops.Z)
        val_B, _ = ns.qubits.measure(qB_rx, ops.Z)
        
        spin_A = 1 if val_A == 0 else -1
        spin_B = 1 if val_B == 0 else -1
        weight = 1.0 if not (idx_A == 1 and idx_B == 1) else -1.0
        products.append(spin_A * spin_B * weight)
        
    return np.mean(products) * 4.0

# =====================================================================
# CENTRALIZED CONTROLLER CORE
# =====================================================================
class CentralNetworkController:
    def __init__(self, topology_edges):
        """Initializes the central controller with a global topology graph layout."""
        self.topology = topology_edges
        self.nodes = sorted(list(set([u for u, v in topology_edges.keys()] + [v for u, v in topology_edges.keys()])))
        self.global_security_matrix = {}

    def discover_network_metrics(self):
        """Controller sweeps the network globally to build a live telemetry map."""
        print("[Central Controller] Initializing global link layer telemetry sweep...")
        for edge, noise in self.topology.items():
            u, v = edge
            S_metric = probe_physical_link(noise)
            self.global_security_matrix[edge] = S_metric
            print(f"  -> Link ({u} -> {v}) | Telemetry: Noise = {noise*100:4.1f}% | Resolved S = {S_metric:.3f}")

    def _find_all_paths(self, start, end, path=[]):
        """Internal graph traversal algorithm to find all loop-free routes."""
        path = path + [start]
        if start == end:
            return [path]
        paths = []
        for edge in self.topology.keys():
            u, v = edge
            if u == start and v not in path:
                extended_paths = self._find_all_paths(v, end, path)
                for p in extended_paths:
                    paths.append(p)
        return paths

    def compute_global_routing(self, source, destination):
        """Executes global bottleneck optimization across the discovered topology graph."""
        self.discover_network_metrics()
        
        all_possible_routes = self._find_all_paths(source, destination)
        print(f"\n[Routing Optimization] Found {len(all_possible_routes)} valid path(s) from {source} to {destination}:")
        
        best_path = None
        highest_bottleneck_S = -1.0
        
        for path in all_possible_routes:
            path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
            bottleneck_S = min([self.global_security_matrix[edge] for edge in path_edges])
            print(f"  Route Profile: {' -> '.join(path)} | End-to-End Bottleneck S = {bottleneck_S:.3f}")
            
            if bottleneck_S > highest_bottleneck_S:
                highest_bottleneck_S = bottleneck_S
                best_path = path

        print("\n" + "="*60)
        if highest_bottleneck_S <= 2.0:
            print(f"[ALERT] CENTRAL COMMAND: ROUTE ADMISSION DENIED.")
            print(f"Reason: Optimal bottleneck path value ({highest_bottleneck_S:.3f}) is below security threshold.")
            print("Action: DROPPING ALL NETWORK KEY INITIALIZATION TRAFFIC.")
            return None
        else:
            print(f"[SUCCESS] CENTRAL COMMAND: PATH ESTABLISHED SUCCESSFULLY.")
            print(f"Selected Route : {' -> '.join(best_path)}")
            print(f"Bottleneck Metric: S = {highest_bottleneck_S:.3f} (> 2.0 Security Cutoff)")
            return best_path

# =====================================================================
# CONFIGURING A 5-NODE MESH NETWORK TOPOLOGY
# =====================================================================
mesh_network_topology = {
    ("Node_A", "Node_B"): 0.02,  # Pristine Link
    ("Node_A", "Node_C"): 0.35,  # Highly Noisy Link
    ("Node_B", "Node_D"): 0.04,  # Pristine Link
    ("Node_C", "Node_D"): 0.05,  # Good Link
    ("Node_B", "Node_E"): 0.45,  # COMPROMISED LINK (Direct route is a trap)
    ("Node_D", "Node_E"): 0.08   # Clean Link
}

controller = CentralNetworkController(mesh_network_topology)
controller.compute_global_routing(source="Node_A", destination="Node_E")

[Central Controller] Initializing global link layer telemetry sweep...
  -> Link (Node_A -> Node_B) | Telemetry: Noise =  2.0% | Resolved S = 2.533
  -> Link (Node_A -> Node_C) | Telemetry: Noise = 35.0% | Resolved S = 1.600
  -> Link (Node_B -> Node_D) | Telemetry: Noise =  4.0% | Resolved S = 2.800
  -> Link (Node_C -> Node_D) | Telemetry: Noise =  5.0% | Resolved S = 2.667
  -> Link (Node_B -> Node_E) | Telemetry: Noise = 45.0% | Resolved S = 1.333
  -> Link (Node_D -> Node_E) | Telemetry: Noise =  8.0% | Resolved S = 3.200

[Routing Optimization] Found 3 valid path(s) from Node_A to Node_E:
  Route Profile: Node_A -> Node_B -> Node_D -> Node_E | End-to-End Bottleneck S = 2.533
  Route Profile: Node_A -> Node_B -> Node_E | End-to-End Bottleneck S = 1.333
  Route Profile: Node_A -> Node_C -> Node_D -> Node_E | End-to-End Bottleneck S = 1.600

[SUCCESS] CENTRAL COMMAND: PATH ESTABLISHED SUCCESSFULLY.
Selected Route : Node_A -> Node_B -> Node_D -> Node_E
Bottleneck Metric: S = 2.533 (>

['Node_A', 'Node_B', 'Node_D', 'Node_E']

## For 7 Node, interconnected mesh
```text
       (Node_B) --------- [0.25] --------- (Node_F)
       /      \                              /    \
   [0.02]    [0.05]                      [0.02]  [0.03]
     /          \                          /        \
(Node_A)        (Node_D) ----------------          (Node_G)
     \          /                          \        /
   [0.10]    [0.04]                      [0.05]  [0.08]
       \      /                              \    /
       (Node_C) --------- [0.30] --------- (Node_E) 

Here I just replace the 5-mesh with 7-mesh which is interconnected; diagram above.

In [6]:
import numpy as np
import netsquid as ns
from netsquid.nodes import Node
from netsquid.components import QuantumChannel
from netsquid.components.models.qerrormodels import DepolarNoiseModel
import netsquid.qubits.operators as ops

# =====================================================================
# PHYSICAL LINK PROBER (Fixed NetSquid Hardware Abstraction)
# =====================================================================
def probe_physical_link(noise_rate, num_trials=60):
    """Simulates physical link layer execution to return an empirical S-value."""
    ns.sim_reset()
    n1, n2 = Node("Tx"), Node("Rx")
    noise_model = DepolarNoiseModel(depolar_rate=noise_rate, time_independent=True)
    channel = QuantumChannel("Link", length=1.0, models={"quantum_noise_model": noise_model})
    
    n1.add_ports(["out"])
    n2.add_ports(["in"])
    n1.ports["out"].connect(channel.ports["send"])
    channel.ports["recv"].connect(n2.ports["in"])
    
    products = []
    alice_angles = [0.0, np.pi / 4]
    bob_angles   = [np.pi / 8, -np.pi / 8]
    
    for _ in range(num_trials):
        qA, qB = ns.qubits.create_qubits(2)
        ns.qubits.operate(qA, ops.H)
        ns.qubits.operate([qA, qB], ops.CNOT)
        n1.ports["out"].tx_output(qB)
        ns.sim_run()
        
        qB_rx = n2.ports["in"].rx_input().items[0]
        idx_A, idx_B = np.random.choice([0, 1]), np.random.choice([0, 1])
        
        # Realignment operators inline
        c_A, s_A = np.cos(-alice_angles[idx_A]), np.sin(-alice_angles[idx_A])
        c_B, s_B = np.cos(-bob_angles[idx_B]), np.sin(-bob_angles[idx_B])
        
        ns.qubits.operate(qA, ops.Operator("RotA", np.array([[c_A, -s_A], [s_A, c_A]])))
        ns.qubits.operate(qB_rx, ops.Operator("RotB", np.array([[c_B, -s_B], [s_B, c_B]])))
        
        val_A, _ = ns.qubits.measure(qA, ops.Z)
        val_B, _ = ns.qubits.measure(qB_rx, ops.Z)
        
        spin_A = 1 if val_A == 0 else -1
        spin_B = 1 if val_B == 0 else -1
        weight = 1.0 if not (idx_A == 1 and idx_B == 1) else -1.0
        products.append(spin_A * spin_B * weight)
        
    return np.mean(products) * 4.0

# =====================================================================
# CENTRALIZED CONTROLLER CORE
# =====================================================================
class CentralNetworkController:
    def __init__(self, topology_edges):
        """Initializes the central controller with a global topology graph layout."""
        self.topology = topology_edges
        self.nodes = sorted(list(set([u for u, v in topology_edges.keys()] + [v for u, v in topology_edges.keys()])))
        self.global_security_matrix = {}

    def discover_network_metrics(self):
        """Controller sweeps the network globally to build a live telemetry map."""
        print("[Central Controller] Initializing global link layer telemetry sweep...")
        for edge, noise in self.topology.items():
            u, v = edge
            S_metric = probe_physical_link(noise)
            self.global_security_matrix[edge] = S_metric
            print(f"  -> Link ({u} -> {v}) | Telemetry: Noise = {noise*100:4.1f}% | Resolved S = {S_metric:.3f}")

    def _find_all_paths(self, start, end, path=[]):
        """Internal graph traversal algorithm to find all loop-free routes."""
        path = path + [start]
        if start == end:
            return [path]
        paths = []
        for edge in self.topology.keys():
            u, v = edge
            if u == start and v not in path:
                extended_paths = self._find_all_paths(v, end, path)
                for p in extended_paths:
                    paths.append(p)
        return paths

    def compute_global_routing(self, source, destination):
        """Executes global bottleneck optimization across the discovered topology graph."""
        self.discover_network_metrics()
        
        all_possible_routes = self._find_all_paths(source, destination)
        print(f"\n[Routing Optimization] Found {len(all_possible_routes)} valid path(s) from {source} to {destination}:")
        
        best_path = None
        highest_bottleneck_S = -1.0
        
        for path in all_possible_routes:
            path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
            bottleneck_S = min([self.global_security_matrix[edge] for edge in path_edges])
            print(f"  Route Profile: {' -> '.join(path)} | End-to-End Bottleneck S = {bottleneck_S:.3f}")
            
            if bottleneck_S > highest_bottleneck_S:
                highest_bottleneck_S = bottleneck_S
                best_path = path

        print("\n" + "="*60)
        if highest_bottleneck_S <= 2.0:
            print(f"[ALERT] CENTRAL COMMAND: ROUTE ADMISSION DENIED.")
            print(f"Reason: Optimal bottleneck path value ({highest_bottleneck_S:.3f}) is below security threshold.")
            print("Action: DROPPING ALL NETWORK KEY INITIALIZATION TRAFFIC.")
            return None
        else:
            print(f"[SUCCESS] CENTRAL COMMAND: PATH ESTABLISHED SUCCESSFULLY.")
            print(f"Selected Route : {' -> '.join(best_path)}")
            print(f"Bottleneck Metric: S = {highest_bottleneck_S:.3f} (> 2.0 Security Cutoff)")
            return best_path

# =====================================================================
# CONFIGURING A 5-NODE MESH NETWORK TOPOLOGY
# =====================================================================
mesh_network_topology = {
    ("Node_A", "Node_B"): 0.02, ("Node_A", "Node_C"): 0.10,
    ("Node_B", "Node_D"): 0.05, ("Node_B", "Node_F"): 0.25,
    ("Node_C", "Node_D"): 0.04, ("Node_C", "Node_E"): 0.30,
    ("Node_D", "Node_E"): 0.05, ("Node_D", "Node_F"): 0.02,
    ("Node_E", "Node_G"): 0.08, ("Node_F", "Node_G"): 0.03
}


controller = CentralNetworkController(mesh_network_topology)
controller.compute_global_routing(source="Node_A", destination="Node_E")

[Central Controller] Initializing global link layer telemetry sweep...
  -> Link (Node_A -> Node_B) | Telemetry: Noise =  2.0% | Resolved S = 3.600
  -> Link (Node_A -> Node_C) | Telemetry: Noise = 10.0% | Resolved S = 2.267
  -> Link (Node_B -> Node_D) | Telemetry: Noise =  5.0% | Resolved S = 3.467
  -> Link (Node_B -> Node_F) | Telemetry: Noise = 25.0% | Resolved S = 2.533
  -> Link (Node_C -> Node_D) | Telemetry: Noise =  4.0% | Resolved S = 2.267
  -> Link (Node_C -> Node_E) | Telemetry: Noise = 30.0% | Resolved S = 1.733
  -> Link (Node_D -> Node_E) | Telemetry: Noise =  5.0% | Resolved S = 2.667
  -> Link (Node_D -> Node_F) | Telemetry: Noise =  2.0% | Resolved S = 3.467
  -> Link (Node_E -> Node_G) | Telemetry: Noise =  8.0% | Resolved S = 2.533
  -> Link (Node_F -> Node_G) | Telemetry: Noise =  3.0% | Resolved S = 2.533

[Routing Optimization] Found 3 valid path(s) from Node_A to Node_E:
  Route Profile: Node_A -> Node_B -> Node_D -> Node_E | End-to-End Bottleneck S = 2.667
  

['Node_A', 'Node_B', 'Node_D', 'Node_E']

### Analysis
It seems that it worked fine

## 15 Node mesh
same code but a larger mesh:
```text
 Layer 1          Layer 2            Layer 3           Layer 4
 (Source)                                           (Destination)

    +-------------> (Node_E) ----------> (Node_I) ---------> (Node_N) --+
   /               /        \          /        \               \
(Node_A) ----> (Node_B) ------> (Node_F) ------> (Node_J) -------> (Node_O)
 |  \         /    |         /        /  \        /    \          ^
 |   +-> (Node_C) -+---> (Node_G) ---+    +-> (Node_K)  +---------+
 |       /         \     /        \       /   /        /
 +-> (Node_D) ------> (Node_H) ------> (Node_L) -> (Node_M) ------+

In [8]:
import numpy as np
import netsquid as ns
from netsquid.nodes import Node
from netsquid.components import QuantumChannel
from netsquid.components.models.qerrormodels import DepolarNoiseModel
import netsquid.qubits.operators as ops

# =====================================================================
# PHYSICAL LINK PROBER (Fixed NetSquid Hardware Abstraction)
# =====================================================================
def probe_physical_link(noise_rate, num_trials=60):
    """Simulates physical link layer execution to return an empirical S-value."""
    ns.sim_reset()
    n1, n2 = Node("Tx"), Node("Rx")
    noise_model = DepolarNoiseModel(depolar_rate=noise_rate, time_independent=True)
    channel = QuantumChannel("Link", length=1.0, models={"quantum_noise_model": noise_model})
    
    n1.add_ports(["out"])
    n2.add_ports(["in"])
    n1.ports["out"].connect(channel.ports["send"])
    channel.ports["recv"].connect(n2.ports["in"])
    
    products = []
    alice_angles = [0.0, np.pi / 4]
    bob_angles   = [np.pi / 8, -np.pi / 8]
    
    for _ in range(num_trials):
        qA, qB = ns.qubits.create_qubits(2)
        ns.qubits.operate(qA, ops.H)
        ns.qubits.operate([qA, qB], ops.CNOT)
        n1.ports["out"].tx_output(qB)
        ns.sim_run()
        
        qB_rx = n2.ports["in"].rx_input().items[0]
        idx_A, idx_B = np.random.choice([0, 1]), np.random.choice([0, 1])
        
        # Realignment operators inline
        c_A, s_A = np.cos(-alice_angles[idx_A]), np.sin(-alice_angles[idx_A])
        c_B, s_B = np.cos(-bob_angles[idx_B]), np.sin(-bob_angles[idx_B])
        
        ns.qubits.operate(qA, ops.Operator("RotA", np.array([[c_A, -s_A], [s_A, c_A]])))
        ns.qubits.operate(qB_rx, ops.Operator("RotB", np.array([[c_B, -s_B], [s_B, c_B]])))
        
        val_A, _ = ns.qubits.measure(qA, ops.Z)
        val_B, _ = ns.qubits.measure(qB_rx, ops.Z)
        
        spin_A = 1 if val_A == 0 else -1
        spin_B = 1 if val_B == 0 else -1
        weight = 1.0 if not (idx_A == 1 and idx_B == 1) else -1.0
        products.append(spin_A * spin_B * weight)
        
    return np.mean(products) * 4.0

# =====================================================================
# CENTRALIZED CONTROLLER CORE
# =====================================================================
class CentralNetworkController:
    def __init__(self, topology_edges):
        """Initializes the central controller with a global topology graph layout."""
        self.topology = topology_edges
        self.nodes = sorted(list(set([u for u, v in topology_edges.keys()] + [v for u, v in topology_edges.keys()])))
        self.global_security_matrix = {}

    def discover_network_metrics(self):
        """Controller sweeps the network globally to build a live telemetry map."""
        print("[Central Controller] Initializing global link layer telemetry sweep...")
        for edge, noise in self.topology.items():
            u, v = edge
            S_metric = probe_physical_link(noise)
            self.global_security_matrix[edge] = S_metric
            print(f"  -> Link ({u} -> {v}) | Telemetry: Noise = {noise*100:4.1f}% | Resolved S = {S_metric:.3f}")

    def _find_all_paths(self, start, end, path=[]):
        """Internal graph traversal algorithm to find all loop-free routes."""
        path = path + [start]
        if start == end:
            return [path]
        paths = []
        for edge in self.topology.keys():
            u, v = edge
            if u == start and v not in path:
                extended_paths = self._find_all_paths(v, end, path)
                for p in extended_paths:
                    paths.append(p)
        return paths

    def compute_global_routing(self, source, destination):
        """Executes global bottleneck optimization across the discovered topology graph."""
        self.discover_network_metrics()
        
        all_possible_routes = self._find_all_paths(source, destination)
        print(f"\n[Routing Optimization] Found {len(all_possible_routes)} valid path(s) from {source} to {destination}:")
        
        best_path = None
        highest_bottleneck_S = -1.0
        
        for path in all_possible_routes:
            path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
            bottleneck_S = min([self.global_security_matrix[edge] for edge in path_edges])
            print(f"  Route Profile: {' -> '.join(path)} | End-to-End Bottleneck S = {bottleneck_S:.3f}")
            
            if bottleneck_S > highest_bottleneck_S:
                highest_bottleneck_S = bottleneck_S
                best_path = path

        print("\n" + "="*60)
        if highest_bottleneck_S <= 2.0:
            print(f"[ALERT] CENTRAL COMMAND: ROUTE ADMISSION DENIED.")
            print(f"Reason: Optimal bottleneck path value ({highest_bottleneck_S:.3f}) is below security threshold.")
            print("Action: DROPPING ALL NETWORK KEY INITIALIZATION TRAFFIC.")
            return None
        else:
            print(f"[SUCCESS] CENTRAL COMMAND: PATH ESTABLISHED SUCCESSFULLY.")
            print(f"Selected Route : {' -> '.join(best_path)}")
            print(f"Bottleneck Metric: S = {highest_bottleneck_S:.3f} (> 2.0 Security Cutoff)")
            return best_path

# ======================================
# COMPLICATED 15-NODE MESH NETWORK TOPOLOGY
# =====================================
mesh_network_topology = {
    # Layer 1 (Source connections)
    ("Node_A", "Node_B"): 0.02, ("Node_A", "Node_C"): 0.05, ("Node_A", "Node_D"): 0.08,
    ("Node_B", "Node_C"): 0.04, ("Node_B", "Node_E"): 0.12, ("Node_B", "Node_F"): 0.15,
    ("Node_C", "Node_D"): 0.03, ("Node_C", "Node_F"): 0.09, ("Node_C", "Node_G"): 0.11,
    ("Node_D", "Node_G"): 0.07, ("Node_D", "Node_H"): 0.14,
    
    # Layer 2 (Mid-mesh backbone)
    ("Node_E", "Node_F"): 0.06, ("Node_E", "Node_I"): 0.05, ("Node_E", "Node_J"): 0.22,
    ("Node_F", "Node_G"): 0.04, ("Node_F", "Node_J"): 0.13, ("Node_F", "Node_K"): 0.08,
    ("Node_G", "Node_H"): 0.05, ("Node_G", "Node_K"): 0.10, ("Node_G", "Node_L"): 0.17,
    ("Node_H", "Node_L"): 0.09, ("Node_H", "Node_M"): 0.25,
    
    # Layer 3 (Destination approaches)
    ("Node_I", "Node_J"): 0.02, ("Node_I", "Node_N"): 0.04,
    ("Node_J", "Node_K"): 0.03, ("Node_J", "Node_N"): 0.07, ("Node_J", "Node_O"): 0.15,
    ("Node_K", "Node_L"): 0.05, ("Node_K", "Node_O"): 0.06,
    ("Node_L", "Node_M"): 0.04, ("Node_L", "Node_O"): 0.08,
    
    # Final Hop
    ("Node_M", "Node_O"): 0.11, ("Node_N", "Node_O"): 0.03
}

# Try running this to see the performance bottleneck:
controller = CentralNetworkController(mesh_network_topology)
controller.compute_global_routing(source="Node_A", destination="Node_O")

controller = CentralNetworkController(mesh_network_topology)
controller.compute_global_routing(source="Node_A", destination="Node_E")

[Central Controller] Initializing global link layer telemetry sweep...
  -> Link (Node_A -> Node_B) | Telemetry: Noise =  2.0% | Resolved S = 2.800
  -> Link (Node_A -> Node_C) | Telemetry: Noise =  5.0% | Resolved S = 3.067
  -> Link (Node_A -> Node_D) | Telemetry: Noise =  8.0% | Resolved S = 2.133
  -> Link (Node_B -> Node_C) | Telemetry: Noise =  4.0% | Resolved S = 2.800
  -> Link (Node_B -> Node_E) | Telemetry: Noise = 12.0% | Resolved S = 2.400
  -> Link (Node_B -> Node_F) | Telemetry: Noise = 15.0% | Resolved S = 2.800
  -> Link (Node_C -> Node_D) | Telemetry: Noise =  3.0% | Resolved S = 2.133
  -> Link (Node_C -> Node_F) | Telemetry: Noise =  9.0% | Resolved S = 2.533
  -> Link (Node_C -> Node_G) | Telemetry: Noise = 11.0% | Resolved S = 2.667
  -> Link (Node_D -> Node_G) | Telemetry: Noise =  7.0% | Resolved S = 2.667
  -> Link (Node_D -> Node_H) | Telemetry: Noise = 14.0% | Resolved S = 3.200
  -> Link (Node_E -> Node_F) | Telemetry: Noise =  6.0% | Resolved S = 2.267
  -> 

['Node_A', 'Node_B', 'Node_E']

### Analysis: Suprisingly, it worked

## 50 Node Mesh

In [9]:
import numpy as np
import netsquid as ns
from netsquid.nodes import Node
from netsquid.components import QuantumChannel
from netsquid.components.models.qerrormodels import DepolarNoiseModel
import netsquid.qubits.operators as ops

# =====================================================================
# PHYSICAL LINK PROBER (Fixed NetSquid Hardware Abstraction)
# =====================================================================
def probe_physical_link(noise_rate, num_trials=60):
    """Simulates physical link layer execution to return an empirical S-value."""
    ns.sim_reset()
    n1, n2 = Node("Tx"), Node("Rx")
    noise_model = DepolarNoiseModel(depolar_rate=noise_rate, time_independent=True)
    channel = QuantumChannel("Link", length=1.0, models={"quantum_noise_model": noise_model})
    
    n1.add_ports(["out"])
    n2.add_ports(["in"])
    n1.ports["out"].connect(channel.ports["send"])
    channel.ports["recv"].connect(n2.ports["in"])
    
    products = []
    alice_angles = [0.0, np.pi / 4]
    bob_angles   = [np.pi / 8, -np.pi / 8]
    
    for _ in range(num_trials):
        qA, qB = ns.qubits.create_qubits(2)
        ns.qubits.operate(qA, ops.H)
        ns.qubits.operate([qA, qB], ops.CNOT)
        n1.ports["out"].tx_output(qB)
        ns.sim_run()
        
        qB_rx = n2.ports["in"].rx_input().items[0]
        idx_A, idx_B = np.random.choice([0, 1]), np.random.choice([0, 1])
        
        # Realignment operators inline
        c_A, s_A = np.cos(-alice_angles[idx_A]), np.sin(-alice_angles[idx_A])
        c_B, s_B = np.cos(-bob_angles[idx_B]), np.sin(-bob_angles[idx_B])
        
        ns.qubits.operate(qA, ops.Operator("RotA", np.array([[c_A, -s_A], [s_A, c_A]])))
        ns.qubits.operate(qB_rx, ops.Operator("RotB", np.array([[c_B, -s_B], [s_B, c_B]])))
        
        val_A, _ = ns.qubits.measure(qA, ops.Z)
        val_B, _ = ns.qubits.measure(qB_rx, ops.Z)
        
        spin_A = 1 if val_A == 0 else -1
        spin_B = 1 if val_B == 0 else -1
        weight = 1.0 if not (idx_A == 1 and idx_B == 1) else -1.0
        products.append(spin_A * spin_B * weight)
        
    return np.mean(products) * 4.0

# =====================================================================
# CENTRALIZED CONTROLLER CORE
# =====================================================================
class CentralNetworkController:
    def __init__(self, topology_edges):
        """Initializes the central controller with a global topology graph layout."""
        self.topology = topology_edges
        self.nodes = sorted(list(set([u for u, v in topology_edges.keys()] + [v for u, v in topology_edges.keys()])))
        self.global_security_matrix = {}

    def discover_network_metrics(self):
        """Controller sweeps the network globally to build a live telemetry map."""
        print("[Central Controller] Initializing global link layer telemetry sweep...")
        for edge, noise in self.topology.items():
            u, v = edge
            S_metric = probe_physical_link(noise)
            self.global_security_matrix[edge] = S_metric
            print(f"  -> Link ({u} -> {v}) | Telemetry: Noise = {noise*100:4.1f}% | Resolved S = {S_metric:.3f}")

    def _find_all_paths(self, start, end, path=[]):
        """Internal graph traversal algorithm to find all loop-free routes."""
        path = path + [start]
        if start == end:
            return [path]
        paths = []
        for edge in self.topology.keys():
            u, v = edge
            if u == start and v not in path:
                extended_paths = self._find_all_paths(v, end, path)
                for p in extended_paths:
                    paths.append(p)
        return paths

    def compute_global_routing(self, source, destination):
        """Executes global bottleneck optimization across the discovered topology graph."""
        self.discover_network_metrics()
        
        all_possible_routes = self._find_all_paths(source, destination)
        print(f"\n[Routing Optimization] Found {len(all_possible_routes)} valid path(s) from {source} to {destination}:")
        
        best_path = None
        highest_bottleneck_S = -1.0
        
        for path in all_possible_routes:
            path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
            bottleneck_S = min([self.global_security_matrix[edge] for edge in path_edges])
            print(f"  Route Profile: {' -> '.join(path)} | End-to-End Bottleneck S = {bottleneck_S:.3f}")
            
            if bottleneck_S > highest_bottleneck_S:
                highest_bottleneck_S = bottleneck_S
                best_path = path

        print("\n" + "="*60)
        if highest_bottleneck_S <= 2.0:
            print(f"[ALERT] CENTRAL COMMAND: ROUTE ADMISSION DENIED.")
            print(f"Reason: Optimal bottleneck path value ({highest_bottleneck_S:.3f}) is below security threshold.")
            print("Action: DROPPING ALL NETWORK KEY INITIALIZATION TRAFFIC.")
            return None
        else:
            print(f"[SUCCESS] CENTRAL COMMAND: PATH ESTABLISHED SUCCESSFULLY.")
            print(f"Selected Route : {' -> '.join(best_path)}")
            print(f"Bottleneck Metric: S = {highest_bottleneck_S:.3f} (> 2.0 Security Cutoff)")
            return best_path


# =====================================================================
# GENERATING A DENSE 50-NODE MESH TOPO (The DFS Bottleneck Test)
# =====================================================================
import random
random.seed(42) 

# rather than manually make the network, we will use a rorw-column function
mesh_network_topology = {}

# let's build a 5 Row x 10 Column lattice grid - total 50 nodes

for row in range(5):
    for col in range(10):
        current_id = row * 10 + col + 1
        current_node = f"Node_{current_id}"
        
        # horizontal cross-connections
        if col < 9:
            neighbor_id = current_id + 1
            mesh_network_topology[(current_node, f"Node_{neighbor_id}")] = round(random.uniform(0.01, 0.15), 3)
            
        # vertical cross-connections
        if row < 4:
            neighbor_id = current_id + 10
            mesh_network_topology[(current_node, f"Node_{neighbor_id}")] = round(random.uniform(0.01, 0.15), 3)
            
        # 3. Diagonal shortcuts (This adds massive loop permutations to freeze DFS)
        if row < 4 and col < 9:
            neighbor_id = current_id + 11
            mesh_network_topology[(current_node, f"Node_{neighbor_id}")] = round(random.uniform(0.05, 0.45), 3)

# Initialize the controller with the 50-node lattice monster
controller = CentralNetworkController(mesh_network_topology)

# Target the opposite corner of the grid
controller.compute_global_routing(source="Node_1", destination="Node_50")

[Central Controller] Initializing global link layer telemetry sweep...
  -> Link (Node_1 -> Node_2) | Telemetry: Noise = 10.0% | Resolved S = 2.400
  -> Link (Node_1 -> Node_11) | Telemetry: Noise =  1.4% | Resolved S = 3.200
  -> Link (Node_1 -> Node_12) | Telemetry: Noise = 16.0% | Resolved S = 2.533
  -> Link (Node_2 -> Node_3) | Telemetry: Noise =  4.1% | Resolved S = 2.933
  -> Link (Node_2 -> Node_12) | Telemetry: Noise = 11.3% | Resolved S = 2.667
  -> Link (Node_2 -> Node_13) | Telemetry: Noise = 32.1% | Resolved S = 2.133
  -> Link (Node_3 -> Node_4) | Telemetry: Noise = 13.5% | Resolved S = 1.867
  -> Link (Node_3 -> Node_13) | Telemetry: Noise =  2.2% | Resolved S = 3.067
  -> Link (Node_3 -> Node_14) | Telemetry: Noise = 21.9% | Resolved S = 2.800
  -> Link (Node_4 -> Node_5) | Telemetry: Noise =  1.4% | Resolved S = 3.200
  -> Link (Node_4 -> Node_14) | Telemetry: Noise =  4.1% | Resolved S = 2.933
  -> Link (Node_4 -> Node_15) | Telemetry: Noise = 25.2% | Resolved S = 1.3

['Node_1',
 'Node_2',
 'Node_3',
 'Node_13',
 'Node_14',
 'Node_15',
 'Node_25',
 'Node_36',
 'Node_37',
 'Node_38',
 'Node_39',
 'Node_40',
 'Node_50']